# SNP Selection — AA Smoking Status (DoubleML + Stability Selection)

**Purpose:** Identify SNPs causally associated with smoking_status via cross-fitted
DoubleML residualization + stability selection, on relatedness-filtered data with
validated (though imperfect — see 02_confounders.ipynb gate) confounder adjustment.

**Inputs:** `checkpoint7b_snp_encoded_012_relatedness_filtered.csv`, `confounders_X.npy`,
`checkpoint2b_metadata_relatedness_filtered.csv`.

**Key parameters:** p < 0.001 per-repeat, 30 repeats, 5-fold cross-fitting.

**Threshold policy:** Given 02_confounders.ipynb's documented λ=1.74 (residual
inflation not fully resolved), this notebook reports BOTH a primary shortlist
(≥80% stability, for cross-cohort/cross-phenotype comparability) and a sensitivity
shortlist (≥100% stability, more resistant to inflation-driven false positives) —
same dual-threshold policy used in EA. Both are checked for near-perfect
inter-SNP correlation (>0.99) before finalizing, since inflation/latent clustering
was found to cause this in EA's primary shortlist (43% removal rate there) — if AA
shows a similarly high removal rate, that strengthens the case for treating the
100% sensitivity set as the trustworthy result, consistent with EA.

**Outputs:** `checkpoint9_doubleml_stability_results.csv`,
`shortlist_smoking_80pct_final.csv`, `shortlist_smoking_100pct_final.csv`.

## Step 1 — Rebuild standardized matrix, load confounders, build outcome

In [1]:
import pandas as pd
import numpy as np
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_auto_64 = X_auto_int.T.astype(np.float64)
del X_auto_int
gc.collect()

p = X_auto_64.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
X_standardized = (X_auto_64[:, valid_snp_mask] - 2 * p[valid_snp_mask]) / denom[valid_snp_mask]
del X_auto_64
gc.collect()

probe_ids_valid = probe_id_array[keep_mask][valid_snp_mask]
print("X_standardized shape:", X_standardized.shape)

X_confounders = np.load(os.path.join(out_dir, "confounders_X.npy"))
print("Confounders shape:", X_confounders.shape)

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
status_map = meta_df.set_index("sample_id")["smoking_status"]
Y = np.array([1.0 if status_map.get(sid) == "Smoker" else 0.0 for sid in sample_ids])
print("Y distribution:", np.unique(Y, return_counts=True))

X_standardized shape: (3036, 141324)
Confounders shape: (3036, 12)
Y distribution: (array([0., 1.]), array([1577, 1459]))


## Step 2 — 30-repeat DoubleML stability selection (p < 0.001 per repeat)

In [2]:
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

n_repeats = 30
threshold = 0.001
n_snps = X_standardized.shape[1]
significant_counts = np.zeros(n_snps, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized, Y, X_confounders, random_state=rep)
    significant_counts += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction = significant_counts / n_repeats
print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction >= t).sum()} SNPs")

Completed 5/30, elapsed 328.6s
Completed 10/30, elapsed 623.1s
Completed 15/30, elapsed 932.0s
Completed 20/30, elapsed 1258.3s
Completed 25/30, elapsed 1569.5s
Completed 30/30, elapsed 1878.3s
Total time: 1878.3s

Stability distribution:
  >= 50%: 1636 SNPs
  >= 60%: 1177 SNPs
  >= 70%: 659 SNPs
  >= 80%: 407 SNPs
  >= 90%: 72 SNPs
  >= 100%: 10 SNPs


## Step 3 — Save both shortlists (primary 80%, sensitivity 100%)

In [3]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
})
stability_df = stability_df.sort_values("stability_fraction", ascending=False)
stability_df.to_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results.csv"), index=False)

shortlist_80 = stability_df[stability_df["stability_fraction"] >= 0.8].copy()
print("Primary shortlist (>=80%):", len(shortlist_80))
shortlist_80.to_csv(os.path.join(out_dir, "shortlist_smoking_80pct_primary.csv"), index=False)

shortlist_100 = stability_df[stability_df["stability_fraction"] >= 1.0].copy()
print("Sensitivity shortlist (=100%):", len(shortlist_100))
shortlist_100.to_csv(os.path.join(out_dir, "shortlist_smoking_100pct_sensitivity.csv"), index=False)

Primary shortlist (>=80%): 407
Sensitivity shortlist (=100%): 10


In [4]:
import pandas as pd
import re

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
position_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

shortlist_100["core_name"] = shortlist_100["probe_id"].map(strip_address_suffix)
shortlist_100_pos = shortlist_100.merge(position_lookup, left_on="core_name", right_index=True, how="left")

print(shortlist_100_pos[["probe_id", "Chr", "MapInfo", "stability_fraction"]].sort_values(["Chr", "MapInfo"]))

                                probe_id Chr      MapInfo  stability_fraction
223             exm2941-0_B_R_1919117890   1    1269488.0                 1.0
232             exm3098-0_T_R_1919138216   1    1277183.0                 1.0
8178          exm100944-0_B_R_1921482882   1  152329460.0                 1.0
11611        exm2277017-0_T_R_1989215336   1  202399880.0                 1.0
91890        exm1003257-0_T_F_1922526121  12   51737607.0                 1.0
112083       exm1245580-0_B_F_2060131617  16   58320066.0                 1.0
122630       exm1379120-0_T_F_1921625489  18   21530071.0                 1.0
57122         exm609218-0_B_F_1918575531   7   22985282.0                 1.0
66933   exm-rs7014346-131_B_R_1990484715   8  128424792.0                 1.0
72311         exm782555-0_B_R_1922302526   9  130206472.0                 1.0


In [5]:
import requests
import time
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

def query_ensembl_grch37(chrom, pos, max_retries=3):
    url = f"https://grch37.rest.ensembl.org/overlap/region/human/{chrom}:{int(pos)-1}-{int(pos)+1}?feature=gene;content-type=application/json"
    headers = {"User-Agent": "Mozilla/5.0 (research script)"}
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(2 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None

gene_map_sensitivity = {}
for _, row in shortlist_100_pos.iterrows():
    snp, chrom, pos = row["probe_id"], row["Chr"], row["MapInfo"]
    data = query_ensembl_grch37(chrom, pos)
    gene_map_sensitivity[snp] = data[0].get("external_name", data[0].get("gene_id", "UNKNOWN")) if data else f"intergenic_chr{chrom}"
    time.sleep(0.3)

for snp, gene in gene_map_sensitivity.items():
    print(f"{snp[:35]:37s} -> {gene}")

with open(os.path.join(out_dir, "gene_map_smoking_sensitivity_grch37.json"), "w") as f:
    json.dump(gene_map_sensitivity, f)
print("Saved.")

exm100944-0_B_R_1921482882            -> FLG-AS1
exm2277017-0_T_R_1989215336           -> PPP1R12B
exm1003257-0_T_F_1922526121           -> CELA1
exm1245580-0_B_F_2060131617           -> PRSS54
exm-rs7014346-131_B_R_1990484715      -> CASC8
exm2941-0_B_R_1919117890              -> TAS1R3
exm782555-0_B_R_1922302526            -> ZNF79
exm3098-0_T_R_1919138216              -> DVL1
exm609218-0_B_F_1918575531            -> FAM126A
exm1379120-0_T_F_1921625489           -> LAMA3
Saved.


In [6]:
import numpy as np

probe_id_to_idx = {pid: i for i, pid in enumerate(probe_ids_valid)}

def get_genotype_vector(probe_id):
    return X_standardized[:, probe_id_to_idx[probe_id]]

# check the two close chr1 SNPs directly first
g1 = get_genotype_vector("exm2941-0_B_R_1919117890")
g2 = get_genotype_vector("exm3098-0_T_R_1919138216")
r2 = np.corrcoef(g1, g2)[0,1] ** 2
print(f"r² between the two chr1:1.27-1.28M SNPs: {r2:.4f}")

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_genotype_vector(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2_pair = np.corrcoef(g_i, get_genotype_vector(pid_j))[0, 1] ** 2
            if r2_pair > r2_thresh:
                removed.add(pid_j)
    return retained

retained_100 = greedy_ld_prune(shortlist_100_pos)
shortlist_100_pruned = shortlist_100_pos[shortlist_100_pos["probe_id"].isin(retained_100)].copy()
print(f"\nSensitivity shortlist after LD pruning: {len(shortlist_100_pruned)} (from {len(shortlist_100_pos)})")
print(shortlist_100_pruned[["probe_id", "Chr", "MapInfo", "stability_fraction"]])

r² between the two chr1:1.27-1.28M SNPs: 0.8995

Sensitivity shortlist after LD pruning: 9 (from 10)
                                probe_id Chr      MapInfo  stability_fraction
8178          exm100944-0_B_R_1921482882   1  152329460.0                 1.0
11611        exm2277017-0_T_R_1989215336   1  202399880.0                 1.0
91890        exm1003257-0_T_F_1922526121  12   51737607.0                 1.0
112083       exm1245580-0_B_F_2060131617  16   58320066.0                 1.0
66933   exm-rs7014346-131_B_R_1990484715   8  128424792.0                 1.0
223             exm2941-0_B_R_1919117890   1    1269488.0                 1.0
72311         exm782555-0_B_R_1922302526   9  130206472.0                 1.0
57122         exm609218-0_B_F_1918575531   7   22985282.0                 1.0
122630       exm1379120-0_T_F_1921625489  18   21530071.0                 1.0


In [7]:
import numpy as np
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

ids_9 = shortlist_100_pruned["probe_id"].tolist()
X_check = np.column_stack([get_genotype_vector(pid) for pid in ids_9])
corr_matrix = np.corrcoef(X_check.T)

to_remove = set()
for i in range(len(ids_9)):
    for j in range(i+1, len(ids_9)):
        if abs(corr_matrix[i,j]) > 0.99 and ids_9[j] not in to_remove:
            to_remove.add(ids_9[j])

print("Perfectly correlated SNPs to remove:", len(to_remove))

shortlist_100_final = shortlist_100_pruned[~shortlist_100_pruned["probe_id"].isin(to_remove)].copy()
print("Final sensitivity shortlist:", len(shortlist_100_final))

shortlist_100_final.to_csv(os.path.join(out_dir, "shortlist_smoking_100pct_final.csv"), index=False)
print("Saved.")

Perfectly correlated SNPs to remove: 0
Final sensitivity shortlist: 9
Saved.


In [8]:
import pandas as pd
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

final_ids = shortlist_100_final["probe_id"].tolist()
print("Final SNP count:", len(final_ids))

final_col_indices = [probe_id_to_idx[pid] for pid in final_ids]
X_pc = X_standardized[:, final_col_indices]

col_names = final_ids + ["smoking_status"]
X_pc_full = np.hstack([X_pc, Y.reshape(-1, 1)])

print("PC input shape:", X_pc_full.shape)

np.save(os.path.join(out_dir, "pc_input_smoking_final.npy"), X_pc_full)
with open(os.path.join(out_dir, "pc_col_names_smoking_final.json"), "w") as f:
    json.dump(col_names, f)
print("Saved.")

Final SNP count: 9
PC input shape: (3036, 10)
Saved.


In [9]:
import pandas as pd
import numpy as np
import re
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
position_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

shortlist_80["core_name"] = shortlist_80["probe_id"].map(strip_address_suffix)
shortlist_80_pos = shortlist_80.merge(position_lookup, left_on="core_name", right_index=True, how="left")
print("Unresolved positions:", shortlist_80_pos["Chr"].isna().sum())

probe_id_to_idx = {pid: i for i, pid in enumerate(probe_ids_valid)}
def get_genotype_vector(probe_id):
    return X_standardized[:, probe_id_to_idx[probe_id]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_genotype_vector(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_genotype_vector(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_80 = greedy_ld_prune(shortlist_80_pos)
shortlist_80_pruned = shortlist_80_pos[shortlist_80_pos["probe_id"].isin(retained_80)].copy()
print("After LD pruning:", len(shortlist_80_pruned))

# now the key diagnostic: perfect-correlation check
ids_pruned = shortlist_80_pruned["probe_id"].tolist()
X_check = np.column_stack([get_genotype_vector(pid) for pid in ids_pruned])
corr_matrix = np.corrcoef(X_check.T)

to_remove = set()
pairs_info = []
for i in range(len(ids_pruned)):
    for j in range(i+1, len(ids_pruned)):
        if abs(corr_matrix[i,j]) > 0.99:
            if ids_pruned[j] not in to_remove:
                to_remove.add(ids_pruned[j])
            row_i = shortlist_80_pruned[shortlist_80_pruned["probe_id"]==ids_pruned[i]].iloc[0]
            row_j = shortlist_80_pruned[shortlist_80_pruned["probe_id"]==ids_pruned[j]].iloc[0]
            same_chr = row_i["Chr"] == row_j["Chr"]
            pairs_info.append(same_chr)

print(f"\nPerfectly correlated SNPs to remove: {len(to_remove)} out of {len(ids_pruned)}")
print(f"Removal rate: {len(to_remove)/len(ids_pruned)*100:.1f}%")
if pairs_info:
    print(f"Pairs on same chromosome: {sum(pairs_info)} / {len(pairs_info)}")
    print(f"Pairs on DIFFERENT chromosomes (latent clustering signature): {len(pairs_info)-sum(pairs_info)} / {len(pairs_info)}")

Unresolved positions: 0
After LD pruning: 361

Perfectly correlated SNPs to remove: 59 out of 361
Removal rate: 16.3%
Pairs on same chromosome: 65 / 1312
Pairs on DIFFERENT chromosomes (latent clustering signature): 1247 / 1312


In [10]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

shortlist_80_final_exploratory = shortlist_80_pruned[~shortlist_80_pruned["probe_id"].isin(to_remove)].copy()
print("Exploratory candidate shortlist (80% stability, cleaned):", len(shortlist_80_final_exploratory))

shortlist_80_final_exploratory.to_csv(os.path.join(out_dir, "shortlist_smoking_80pct_exploratory_final.csv"), index=False)
print("Saved.")

Exploratory candidate shortlist (80% stability, cleaned): 302
Saved.


In [11]:
import pandas as pd
import numpy as np
import re
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# reload full stability results
stability_df = pd.read_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results.csv"))

shortlist_90 = stability_df[stability_df["stability_fraction"] >= 0.9].copy()
print("Shortlist (>=90% stability):", len(shortlist_90))

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
position_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

shortlist_90["core_name"] = shortlist_90["probe_id"].map(strip_address_suffix)
shortlist_90_pos = shortlist_90.merge(position_lookup, left_on="core_name", right_index=True, how="left")
print("Unresolved positions:", shortlist_90_pos["Chr"].isna().sum())

Shortlist (>=90% stability): 72
Unresolved positions: 0


In [12]:
import pandas as pd
import numpy as np
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df2 = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df2.loc[manifest_df2["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df2
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_auto_64 = X_auto_int.T.astype(np.float64)
del X_auto_int
gc.collect()

p = X_auto_64.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
X_standardized = (X_auto_64[:, valid_snp_mask] - 2 * p[valid_snp_mask]) / denom[valid_snp_mask]
del X_auto_64
gc.collect()

probe_ids_valid = probe_id_array[keep_mask][valid_snp_mask]
probe_id_to_idx = {pid: i for i, pid in enumerate(probe_ids_valid)}
print("X_standardized shape:", X_standardized.shape)

def get_genotype_vector(probe_id):
    return X_standardized[:, probe_id_to_idx[probe_id]]

X_standardized shape: (3036, 141324)


In [13]:
import numpy as np

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_genotype_vector(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_genotype_vector(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_90 = greedy_ld_prune(shortlist_90_pos)
shortlist_90_pruned = shortlist_90_pos[shortlist_90_pos["probe_id"].isin(retained_90)].copy()
print("After LD pruning:", len(shortlist_90_pruned))

ids_90 = shortlist_90_pruned["probe_id"].tolist()
X_check_90 = np.column_stack([get_genotype_vector(pid) for pid in ids_90])
corr_90 = np.corrcoef(X_check_90.T)

to_remove_90 = set()
cross_chr_count = 0
same_chr_count = 0
for i in range(len(ids_90)):
    for j in range(i+1, len(ids_90)):
        if abs(corr_90[i,j]) > 0.99:
            if ids_90[j] not in to_remove_90:
                to_remove_90.add(ids_90[j])
            row_i = shortlist_90_pruned[shortlist_90_pruned["probe_id"]==ids_90[i]].iloc[0]
            row_j = shortlist_90_pruned[shortlist_90_pruned["probe_id"]==ids_90[j]].iloc[0]
            if row_i["Chr"] == row_j["Chr"]:
                same_chr_count += 1
            else:
                cross_chr_count += 1

print(f"\nPerfectly correlated SNPs to remove: {len(to_remove_90)} out of {len(ids_90)}")
print(f"Same-chr pairs: {same_chr_count}, cross-chr pairs: {cross_chr_count}")

shortlist_90_final = shortlist_90_pruned[~shortlist_90_pruned["probe_id"].isin(to_remove_90)].copy()
print(f"\nFinal 90% shortlist: {len(shortlist_90_final)}")

import os
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
shortlist_90_final.to_csv(os.path.join(out_dir, "shortlist_smoking_90pct_final.csv"), index=False)
print("Saved.")

After LD pruning: 67

Perfectly correlated SNPs to remove: 0 out of 67
Same-chr pairs: 0, cross-chr pairs: 0

Final 90% shortlist: 67
Saved.
